# Policy `occ_gate`

Bỏ chạy giờ PI vào ngày tòa nhà không có người (ngày lễ).

Xây trên policy tham chiếu (public LB 0,01582). Notebook tự chứa toàn bộ code.

| Chạy ở | Làm gì | Ra |
|---|---|---|
| Máy local | Chạy 30 kịch bản train, 8.640 bước mỗi kịch bản | Điểm train, so với `pid_water` và bản tham chiếu |
| Kaggle (Save & Run All) | Như trên + chạy inference server | Như trên, và **Submit** được để lấy điểm LB |

Khi Kaggle chạy lại để chấm (submit), phần đánh giá train tự bỏ qua, chỉ còn server.

In [ ]:
import os, sys, json, time
from pathlib import Path

for k in ("NUMBA_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[k] = "1"   # one thread per simulator; parallelism = scenarios side by side
# os.environ["SMART_BUILDINGS_DATA_ROOT"] = "/path/to/starter-kit"   # if not found automatically

IS_RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN") is not None
ON_KAGGLE = Path("/kaggle/working").exists()
WORKERS = int(os.getenv("SB_WORKERS", 0)) or os.cpu_count()   # best: number of physical cores

POLICY = "occ_gate"
N_STEPS = 8640            # full 30-day episode; shorter runs are not comparable
ONLY = []                 # e.g. ["building_train_8/phoenix_4"] for a quick check; [] = all 30
RUN_EVAL = not IS_RERUN   # score on the training scenarios
RUN_REF = False           # also run the reference on the same scenarios (doubles the time)
REF_SCORES_CSV = ""       # or: scores_ref.csv from notebook 00, to compare without re-running
RUN_SERVER = ON_KAGGLE    # Kaggle: serve this policy so the version can be submitted
OUT = Path.cwd().resolve() / "results" / POLICY
OUT.mkdir(parents=True, exist_ok=True)

## Code

Ba file được ghi ra cạnh notebook, để các tiến trình chạy song song import được.

In [ ]:
%%writefile sb_paths.py
"""Locate the starter kit (or Kaggle competition data) and make it importable.

Search order: $SMART_BUILDINGS_DATA_ROOT, Kaggle input folders, then a
`starter-kit` folder in, next to, or above the current/code directory.
"""
import os
import sys
from pathlib import Path

CODE_DIR = Path(__file__).resolve().parent
REPO_ROOT = CODE_DIR.parent


def _candidates():
    return [
        os.getenv("SMART_BUILDINGS_DATA_ROOT", ""),
        "/kaggle/input/competitions/neurips-2026-smart-buildings",
        "/kaggle/input/neurips-2026-smart-buildings",
        str(Path.cwd() / "starter-kit"),
        str(Path.cwd().parent / "starter-kit"),
        str(Path.cwd().parent.parent / "starter-kit"),
        str(REPO_ROOT / "starter-kit"),
        str(REPO_ROOT.parent / "starter-kit"),
    ]


def find_data_root() -> Path:
    for p in _candidates():
        if p and (Path(p) / "smart_buildings").exists():
            return Path(p).resolve()
    raise FileNotFoundError(
        "Starter kit not found. Attach the competition data on Kaggle, clone "
        "https://github.com/smart-buildings-2026/starter-kit next to this folder, "
        "or set SMART_BUILDINGS_DATA_ROOT.")


def setup(single_thread: bool = True) -> Path:
    """Make the data root and this code importable here and in child processes."""
    if single_thread:
        for k in ("NUMBA_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
            os.environ.setdefault(k, "1")
    root = find_data_root()
    os.environ["SMART_BUILDINGS_DATA_ROOT"] = str(root)
    # Absolute paths: workers chdir into the data root, so '' would stop working.
    for p in (str(REPO_ROOT), str(CODE_DIR), str(root)):
        if p not in sys.path:
            sys.path.insert(0, p)
    return root


def scenario_dir(scenario) -> Path:
    """'building_train_1/atlanta_2' -> training scenario folder; an existing path is returned as is."""
    p = Path(scenario)
    if p.is_absolute() and (p / "config.gin").exists():
        return p
    return find_data_root() / "smart_buildings" / "training_scenarios" / scenario


def all_scenarios() -> list:
    """Every scenario folder present, including ones you generated."""
    root = find_data_root() / "smart_buildings" / "training_scenarios"
    return sorted(f"{d.parent.name}/{d.name}" for d in root.glob("*/*") if (d / "config.gin").exists())


def training_scenarios() -> list:
    """The 30 official training scenarios (from training_set.yaml when present)."""
    present = all_scenarios()
    manifest = find_data_root() / "smart_buildings" / "training_set.yaml"
    if manifest.exists():
        import yaml
        pairs = yaml.safe_load(manifest.read_text()).get("scenarios", [])
        official = {f"{p['building']}/{p['weather']}" for p in pairs}
        if official:
            return [s for s in present if s in official]
    return present


In [ ]:
%%writefile sb_policy.py
"""Policies: the reference controller (public LB 0.01582, copied verbatim from the Kaggle
notebook "Smart Buildings Probe DS K07", Apache-2.0) plus candidate versions.
Generated from sbx/ref_policy.py and sbx/candidates.py."""
import ast
import numpy as np
from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy


class BaselinePolicy(PIDWaterResetPolicy):
    USE_SENSOR_BAND = True
    # Worst-zone aggregation increased loss in both full-length pilot scenarios.
    USE_WORST_ZONE = False

    @property
    def name(self):
        return "transfer"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        stats = self.observation_normalization_metadata.get(
            "outside_air_temperature_sensor", {}
        )
        self.outside_mean = float(stats.get("sample_mean", 0.0))
        self.outside_std = float(stats.get("sample_std", 1.0))

    def _recirculation(self, obs, ctrl, u, low, high):
        # The parent's economizer expects kelvin; the observation is standardized.
        idx = ctrl.get("oat_idx")
        if idx is None:
            return 1.0
        decoded = obs.copy()
        decoded[idx] = obs[idx] * self.outside_std + self.outside_mean
        return super()._recirculation(decoded, ctrl, u, low, high)

    def get_action(self, obs):
        action = super().get_action(obs)
        # Enforce the gateway's dtype, shape and bounds contract.
        for key, dtype in (("continuous_action", np.float32),
                           ("discrete_action", np.int32)):
            space = self.action_spec[key]
            action[key] = np.clip(action[key], space.low, space.high).astype(dtype)
        return action


class CoolingHoldPolicy(BaselinePolicy):
    HOLD_STEPS = 288
    # Outside work hours use (eco low, comfort high): no overnight drift up to
    # the eco ceiling in hot climates, full heating setback kept in cold ones.
    # Idea from the public "Neurips 2026 Smart Buildings Smartpolicy" notebook.
    NIGHT_COOL_CEILING = False

    @property
    def name(self):
        return f"cooling_hold_{self.HOLD_STEPS}"

    def _base_action(self, obs):
        if not self.NIGHT_COOL_CEILING:
            return super().get_action(obs)
        hour, day = self._get_hour_and_day(obs)
        if day < 5 and self.morning_start_hour <= hour < self.evening_start_hour:
            return super().get_action(obs)
        saved = self.eco_temp_window
        self.eco_temp_window = (saved[0], self.comfort_temp_window[1])
        try:
            return super().get_action(obs)
        finally:
            self.eco_temp_window = saved

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._on_steps = [0] * len(self.ahu_controllers)
        self._last_on = [None] * len(self.ahu_controllers)

    def get_action(self, obs):
        action = self._base_action(obs)
        cont, disc = action["continuous_action"], action["discrete_action"]
        for i, ctrl in enumerate(self.ahu_controllers):
            if not ctrl["zone_obs_indices"]:
                continue
            run = ctrl["run_discrete_idx"]
            # Supply temperature first: its sign tells cooling from heating.
            idx = [j for j in (ctrl["temp_idx"], ctrl["pressure_idx"], ctrl.get("recirc_idx"))
                   if j is not None]
            if (not disc[run] and self._on_steps[i] > 0
                    and self._on_steps[i] < self.HOLD_STEPS and self._last_on[i][0] < 0):
                disc[run] = 1
                cont[idx] = self._last_on[i]
            if disc[run]:
                self._on_steps[i] += 1
                self._last_on[i] = cont[idx].copy()
            else:
                self._on_steps[i] = 0
        return action


class ZonePIPolicy(CoolingHoldPolicy):
    """v5 controller with per-zone PI control to the band centre in work hours.

    A soft proportional zone loop stops the 2-4 K oscillation but leaves zones
    sitting steadily outside the band (no integral action). Here each zone runs
    a PI loop on (temperature - band centre) from one hour before occupancy to
    the end of the work day. One signal u in [-1, 1] drives both actuators:
    the damper opens with |u| (air is needed to cool and to carry reheat) and
    the reheat valve opens with -u. AHUs are kept running in those hours so the
    zones have air to work with. Outside them, v5 logic is unchanged.
    """
    HOLD_STEPS = 288
    NIGHT_COOL_CEILING = True
    KP = 0.2
    KI = 0.02
    PRESTART_H = 1
    # Hours before the end of the work day at which the PI loops stop (coast).
    EVENING_STOP_H = 0
    FORCED_SAT = -0.7       # normalized supply temp for a forced-on AHU (~15 C)
    FORCED_PRESSURE = -0.6  # normalized static pressure for a forced-on AHU
    KEY = "zone_pi"

    @property
    def name(self):
        return self.KEY

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._zone_integral = {}
        self._zone_u = {}
        self._pi_active = False

    def _pi_hours(self, obs):
        hour, day = self._get_hour_and_day(obs)
        return (day < 5 and self.morning_start_hour - self.PRESTART_H <= hour
                < self.evening_start_hour - self.EVENING_STOP_H)

    def _command_zones(self, continuous, zones, obs, low, high):
        if not self._pi_active:
            for vav_id, _, _ in zones:
                self._zone_integral[vav_id] = 0.0
            return super()._command_zones(continuous, zones, obs, low, high)
        lo_c, hi_c = self.comfort_temp_window
        for vav_id, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            lo, hi = self._zone_band(obs, vav_id, lo_c, hi_c)
            # Before occupancy the sensor band is still the eco band; aim at comfort.
            if hi - lo > (hi_c - lo_c) + 1e-6:
                lo, hi = lo_c, hi_c
            self._cur_vav = vav_id
            err = self._denormalize_zone_temp(obs[obs_idx]) - self._zone_target(obs, lo, hi)
            integral = self._zone_integral.get(vav_id, 0.0)
            u_raw = self.KP * err + self.KI * integral
            # Conditional integration: stop winding up while saturated.
            integral *= 1.0 - self.INTEGRAL_LEAK
            if abs(u_raw) < 1.0 or np.sign(err) != np.sign(u_raw):
                limit = self.INTEGRAL_LIMIT_FRAC / self.KI
                integral = float(np.clip(integral + err, -limit, limit))
            self._zone_integral[vav_id] = integral
            u = float(np.clip(self.KP * err + self.KI * integral, -1.0, 1.0))
            if u < 0:
                u *= self.HEAT_U_SCALE
            self._max_heat = max(self._max_heat, -u)
            self._zone_u[vav_id] = u
            gain = self.HEAT_DAMPER_GAIN if u < 0 else 1.0
            damper = self.MIN_DAMPER + (1.0 - self.MIN_DAMPER) * abs(u) * gain
            reheat = max(0.0, -u)
            if self.REHEAT_FULL_AT is not None and u < 0:
                # Hottest possible discharge for the air that is moved; the
                # damper alone sets how much heat reaches the zone.
                reheat = min(1.0, -u / self.REHEAT_FULL_AT)
            split = self.HEAT_SEQUENCE_SPLIT
            if split is not None and u < 0:
                # Reheat first at minimum air; add air only once reheat is full.
                reheat = min(1.0, -u / split)
                damper = self.MIN_DAMPER + (1.0 - self.MIN_DAMPER) * max(0.0, (-u - split) / (1.0 - split))
            damper_idx = action_indices.get("supply_air_damper_percentage_command")
            if damper_idx is not None:
                continuous[damper_idx] = 2.0 * damper - 1.0
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is not None:
                continuous[reheat_idx] = 2.0 * reheat - 1.0

    MIN_DAMPER = 0.1
    # Damper opening per unit of heating demand (reheated air costs gas).
    HEAT_DAMPER_GAIN = 1.0
    # Anti-windup: integral term limited to this share of full output, and a
    # per-step leak towards zero (both neutral by default).
    INTEGRAL_LIMIT_FRAC = 1.0
    INTEGRAL_LEAK = 0.0
    # Heating sequence: None = damper and reheat both follow |u|; a number s =
    # reheat opens fully by |u| = s at minimum air, then the damper opens.
    HEAT_SEQUENCE_SPLIT = None
    # Reheat valve fully open once heating demand reaches this |u| (None =
    # valve follows |u|): each unit of cold mixed air arrives as hot as allowed.
    REHEAT_FULL_AT = None
    # Scale of the PI output when heating (softer loop for hot-discharge heating).
    HEAT_U_SCALE = 1.0

    def _zone_target(self, obs, lo, hi):
        """Temperature each zone's PI loop aims at; the band centre by default."""
        return 0.5 * (lo + hi)

    def _base_action(self, obs):
        self._pi_active = self._pi_hours(obs)
        self._max_heat = 0.0
        self._forced = []
        self._zone_u = {}
        action = super()._base_action(obs)
        if self._pi_active:
            cont, disc = action["continuous_action"], action["discrete_action"]
            # Zone reheat needs hot water even when no AHU loop asks for heat.
            if self.water_temp_idx is not None and self._max_heat > 0:
                cont[self.water_temp_idx] = max(cont[self.water_temp_idx],
                                                np.float32(2.0 * self._max_heat - 1.0))
                if self.water_pressure_idx is not None:
                    cont[self.water_pressure_idx] = 1.0
            for i, ctrl in enumerate(self.ahu_controllers):
                if ctrl["zone_obs_indices"] and not disc[ctrl["run_discrete_idx"]]:
                    self._forced.append(i)
                    disc[ctrl["run_discrete_idx"]] = 1
                    cont[ctrl["temp_idx"]] = self.FORCED_SAT
                    if ctrl["pressure_idx"] is not None:
                        cont[ctrl["pressure_idx"]] = self.FORCED_PRESSURE
            if 0 <= self.water_discrete_idx < self.n_discrete:
                disc[self.water_discrete_idx] = 1
        return action

    def get_action(self, obs):
        action = super().get_action(obs)
        # A forced-on unit is not a cooling run; do not let the hold extend it.
        for i in self._forced:
            if self._last_on[i] is not None:
                self._last_on[i][0] = 0.0
        return action


def _outside_kelvin(policy, value):
    """AHU outside-air sensors arrive in raw kelvin; decode only if normalized."""
    value = float(value)
    return value if value > 150.0 else value * policy.outside_std + policy.outside_mean


class ZonePIEconFixPolicy(ZonePIPolicy):
    """pe_lowp with a working economizer (raw-kelvin outside air temperature).

    TransferPolicy decoded the AHU outside-air sensor as if normalized, but it
    arrives in kelvin, so the economizer saw ~3000 K outside and never opened.
    """
    KP = 0.10
    KI = 0.005
    FORCED_PRESSURE = -1.0
    WARM_SAT = None
    WARM_BELOW_K = 283.15
    KEY = "econ_fix"

    def _recirculation(self, obs, ctrl, u, low, high):
        idx = ctrl.get("oat_idx")
        if idx is None:
            return 1.0
        fixed = obs.copy()
        fixed[idx] = _outside_kelvin(self, obs[idx])
        # Skip TransferPolicy's decoding; call the organizer economizer directly.
        return super(BaselinePolicy, self)._recirculation(fixed, ctrl, u, low, high)

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if self.WARM_SAT is None:
            return action
        cont = action["continuous_action"]
        for i in self._forced:
            ctrl = self.ahu_controllers[i]
            idx = ctrl.get("oat_idx")
            if idx is not None and _outside_kelvin(self, obs[idx]) < self.WARM_BELOW_K:
                cont[ctrl["temp_idx"]] = self.WARM_SAT
        return action


class DemandSatPolicy(ZonePIEconFixPolicy):
    """Supply-air temperature follows the zones' own PI demand in work hours.

    Forced-on AHUs used a fixed ~15 C supply. In cool weather that chills an
    already cool mix, then zone reheat burns gas to warm it back. Here each
    running AHU looks at the mean PI output of its zones (positive = cooling):
    with net heating demand the supply setpoint goes to maximum (no chiller);
    with cooling demand it ramps down to ~15 C, reached at mean demand RAMP
    (RAMP 0 = switch at zero demand).
    """
    RAMP = 0.0
    COLD_SAT = -0.7
    KEY = "dsat_bin"

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if not self._pi_active:
            return action
        cont, disc = action["continuous_action"], action["discrete_action"]
        for ctrl in self.ahu_controllers:
            us = [self._zone_u[v] for v, _, _ in ctrl["zones"] if v in self._zone_u]
            if not us or not disc[ctrl["run_discrete_idx"]]:
                continue
            d = float(np.mean(us))
            frac = (float(np.clip(d / self.RAMP, 0.0, 1.0)) if self.RAMP > 0
                    else (1.0 if d > 0 else 0.0))
            cont[ctrl["temp_idx"]] = np.float32(1.0 + frac * (self.COLD_SAT - 1.0))
        return action


class TunedPIPolicy(ZonePIEconFixPolicy):
    """econ_fix_warmcold plus switchable energy measures, all off by default.

    NIGHT_HEAT_DROP_K: lower the heating setpoint outside PI hours.
    WATER_SCALE: scale the hot-water temperature command (0..1 of range).
    TARGET_SHIFT_K: aim zones this far from the band centre towards the side
        the weather pushes them to (cheaper to hold).
    OCC_GATE: after the arrival hour, run PI hours only while the building
        reports occupants (skips holidays and empty days).
    """
    WARM_SAT = -0.4
    NIGHT_HEAT_DROP_K = 0.0
    WATER_SCALE = 1.0
    TARGET_SHIFT_K = 0.0
    OCC_GATE = False
    # Shift the target towards the low edge only in heating weather.
    COLD_SHIFT_ONLY = False
    # In PI hours, run the hot-water loop at full temperature whenever any
    # zone calls for heat (more heat per unit of cold mixed air).
    WATER_MAX_PI = False
    KEY = "tuned"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        names = list(self.field_names)
        self._occ_idx = names.index("num_occupants") if "num_occupants" in names else None
        self._oat_any = next((c["oat_idx"] for c in self.ahu_controllers
                              if c.get("oat_idx") is not None), None)

    def _pi_hours(self, obs):
        active = super()._pi_hours(obs)
        if active and self.OCC_GATE and self._occ_idx is not None:
            hour, _ = self._get_hour_and_day(obs)
            if hour >= self.morning_start_hour + 2 and float(obs[self._occ_idx]) < 0.5:
                return False
        return active

    def _zone_target(self, obs, lo, hi):
        centre = 0.5 * (lo + hi)
        if self.TARGET_SHIFT_K <= 0 or self._oat_any is None:
            return centre
        outside = _outside_kelvin(self, obs[self._oat_any])
        shift = min(self.TARGET_SHIFT_K, 0.4 * (hi - lo))
        if outside < centre:
            return centre - shift
        return centre if self.COLD_SHIFT_ONLY else centre + shift

    def _zone_band(self, obs, vav_id, low, high):
        lo, hi = super()._zone_band(obs, vav_id, low, high)
        if self.NIGHT_HEAT_DROP_K and not self._pi_active:
            lo -= self.NIGHT_HEAT_DROP_K
        return lo, hi

    def _ahu_error(self, obs, ctrl, low, high):
        if self.NIGHT_HEAT_DROP_K and not self._pi_active:
            low -= self.NIGHT_HEAT_DROP_K
        return super()._ahu_error(obs, ctrl, low, high)

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if (self.WATER_MAX_PI and self._pi_active and self.water_temp_idx is not None
                and self._max_heat > 0):
            action["continuous_action"][self.water_temp_idx] = np.float32(1.0)
        if self.WATER_SCALE != 1.0 and self.water_temp_idx is not None:
            cont = action["continuous_action"]
            frac = (float(cont[self.water_temp_idx]) + 1.0) * 0.5 * self.WATER_SCALE
            cont[self.water_temp_idx] = np.float32(2.0 * frac - 1.0)
        return action


class TunedDemandSatPolicy(TunedPIPolicy, DemandSatPolicy):
    """dsat_bin with the TunedPIPolicy switches (all off by default)."""
    WARM_SAT = None  # dsat_bin did not use the cold-weather warm supply
    KEY = "tuned_dsat"


class AdaptiveColdPolicy(TunedDemandSatPolicy):
    """Pick the heating measures from a smoothed outside temperature.

    Screening showed each cold-weather measure helps one regime and hurts
    another: an early start and full-temperature water with less heating air
    pay off only in deep cold (mean -21 C), while aiming 0.5 K below the band
    centre pays off in moderate cold but hurts deep cold. The regime follows an
    exponential average of outside air temperature over about a day.
    """
    NIGHT_HEAT_DROP_K = 2.0
    PRESTART_H = 2
    NIGHT_COOL_CEILING = False
    EXTREME_BELOW_C = -15.0
    MILD_BELOW_C = 12.0
    EXTREME_PRESTART_H = 4
    EXTREME_WATER_MAX = True
    EXTREME_HEAT_DAMPER = 0.5
    MILD_SHIFT_K = 0.5
    EMA_STEPS = 288
    # Full hot-water temperature in PI hours whenever a zone heats, any weather.
    ALWAYS_WATER_MAX = False
    KEY = "adaptive_cold"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._oat_ema = None
        self._base_prestart = self.PRESTART_H

    def _update_regime(self, obs):
        if self._oat_any is not None:
            outside = _outside_kelvin(self, obs[self._oat_any]) - 273.15
            if self._oat_ema is None:
                self._oat_ema = outside
            else:
                self._oat_ema += (outside - self._oat_ema) / self.EMA_STEPS
        ema = self._oat_ema
        extreme = ema is not None and ema < self.EXTREME_BELOW_C
        mild = ema is not None and not extreme and ema < self.MILD_BELOW_C
        self.PRESTART_H = self.EXTREME_PRESTART_H if extreme else self._base_prestart
        self.WATER_MAX_PI = bool((extreme and self.EXTREME_WATER_MAX) or self.ALWAYS_WATER_MAX)
        self.HEAT_DAMPER_GAIN = self.EXTREME_HEAT_DAMPER if extreme else 1.0
        self.TARGET_SHIFT_K = self.MILD_SHIFT_K if mild else 0.0
        self.COLD_SHIFT_ONLY = True

    def _base_action(self, obs):
        self._update_regime(obs)
        return super()._base_action(obs)


# ---------------------------------------------------------------------------
# Additions for offline evaluation (not part of the reference notebook).
# ---------------------------------------------------------------------------

class RefPolicy(AdaptiveColdPolicy):
    """SubmissionPolicy of the reference notebook, minus the server base class."""
    KP = 0.1
    KI = 0.005
    HOLD_STEPS = 288
    NIGHT_COOL_CEILING = False
    EXTREME_WATER_MAX = False
    EXTREME_HEAT_DAMPER = 1.0
    KEY = "ref"

# --- candidate versions --------------------------------------------------
class _TimeMixin:
    def _frac_hour(self, obs) -> float:
        angle = float(np.arctan2(obs[self.hod_sin_index], obs[self.hod_cos_index]))
        if angle < 0:
            angle += 2 * np.pi
        return 24.0 * angle / (2 * np.pi)

    def _zone_temps(self, ctrl, obs):
        return [self._denormalize_zone_temp(obs[j]) for j in ctrl["zone_obs_indices"]]

class NudgedPolicy(RefPolicy):
    """Reference plus a physically negligible pressure offset: a noise probe."""
    NUDGE = 1e-5

    def get_action(self, obs):
        action = super().get_action(obs)
        cont = action["continuous_action"]
        for ctrl in self.ahu_controllers:
            j = ctrl.get("pressure_idx")
            if j is not None:
                cont[j] = np.float32(np.clip(cont[j] + self.NUDGE, -1.0, 1.0 - 1e-6))
        return action

class CandidatePolicy(RefPolicy):
    """Policy version `occ_gate`: the one this notebook scores and submits."""
    OCC_GATE = True

def _baseline(name):
    from smart_buildings.baselines.bang_bang import BangBangPolicy
    from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy
    return {"bang_bang": BangBangPolicy, "pid_water": PIDWaterResetPolicy}[name]

def parse_spec(spec: str):
    name, _, rest = spec.partition(":")
    overrides = {}
    for part in filter(None, rest.split(",")):
        k, _, v = part.partition("=")
        overrides[k.strip()] = ast.literal_eval(v.strip())
    return name.strip(), overrides

def make(spec: str, building_meta_data, action_spec):
    """Instantiate a policy from a spec string (see module docstring)."""
    name, overrides = parse_spec(spec)
    if name in ("bang_bang", "pid_water"):
        return _baseline(name)(building_meta_data, action_spec)
    if name == "rl":  # trained network, e.g. "rl:WEIGHTS='runs/ppo/best.pt'"
        from sbx.rl.policy import NeuralPolicy
        cls, base = NeuralPolicy, {}
    elif name.startswith("null_"):
        cls, base = NudgedPolicy, {"NUDGE": int(name.split("_")[1]) * 1e-5}
    elif name in REGISTRY:
        cls, base = REGISTRY[name]
    else:
        raise ValueError(f"unknown policy {name!r}; known: {sorted(REGISTRY)}")
    attrs = {**base, **overrides}
    for k in attrs:
        if not hasattr(cls, k):
            raise AttributeError(f"{cls.__name__} has no attribute {k!r}")
    if attrs:
        cls = type(f"{cls.__name__}_custom", (cls,), attrs)
    return cls(building_meta_data, action_spec)

REGISTRY = {
    'ref': (RefPolicy, {}),
    'occ_gate': (RefPolicy, {'OCC_GATE': True}),
    'cand': (CandidatePolicy, {}),
}
CANDIDATES = ['occ_gate']


In [ ]:
%%writefile sb_eval.py
"""Roll out a policy on scenarios, in parallel, with cached results.

A *policy spec* names a policy in `sbx.candidates.REGISTRY`, optionally with
class-attribute overrides:  "ref", "fan_tr", "ref:KP=0.12,KI=0.006".
Results go to <out>/<spec-tag>/<building>__<weather>.json and are reused when
the same spec, scenario and step count were already run.

The rollout loop mirrors the starter kit's evaluate.py (SimulationTracker,
occupancy constant 125, validate_action), so returns are comparable with
baseline_returns.json.
"""
import hashlib
import json
import multiprocessing as mp
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np

import sb_paths as paths

STEPS_PER_DAY = 288
FULL_EPISODE = 8640


def spec_tag(spec: str) -> str:
    """Filesystem-safe folder name for a policy spec."""
    safe = "".join(c if c.isalnum() or c in "._-=," else "_" for c in spec)
    if len(safe) <= 80:
        return safe
    return safe[:60] + "_" + hashlib.sha1(spec.encode()).hexdigest()[:10]


def scenario_key(scenario: str) -> str:
    p = Path(scenario)
    return f"{p.parent.name}__{p.name}"


def result_path(out_dir, spec, scenario) -> Path:
    return Path(out_dir) / spec_tag(spec) / (scenario_key(scenario) + ".json")


def load_env(scenario):
    """Load one scenario's environment exactly as evaluate.py does."""
    root = paths.setup()
    os.chdir(root)  # gin files use paths relative to the data root
    from smart_buildings.evaluate import create_building_meta_data, load_hybrid_environment
    sdir = paths.scenario_dir(scenario)
    env = load_hybrid_environment(sdir / "config.gin")
    env.metrics_path = None
    env._occupancy_normalization_constant = 125.0
    meta = create_building_meta_data(env, sdir)
    return env, meta


def run_scenario(scenario: str, spec: str, n_steps: int = FULL_EPISODE, out_dir=None,
                 save_steps: bool = False) -> dict:
    """Run one policy spec on one scenario; return (and optionally save) the summary."""
    paths.setup()
    from smart_buildings.evaluate import validate_action
    from smart_buildings.smart_control.utils.scenario_generator import SimulationTracker
    import sb_policy as candidates

    t_start = time.perf_counter()
    env, meta = load_env(scenario)
    tracker = SimulationTracker(env, vmin=280, vmax=310, capture_snapshots=False)
    spec_obj = env.action_spec()
    policy = candidates.make(spec, meta, spec_obj)

    tracker.reset()
    obs = tracker.last_obs
    t_policy = t_env = 0.0
    for _ in range(n_steps):
        t0 = time.perf_counter()
        action = validate_action(policy.get_action(obs), spec_obj)
        t1 = time.perf_counter()
        obs, _, terminated, truncated, _ = tracker.step(action)
        t_policy += t1 - t0
        t_env += time.perf_counter() - t1
        if terminated or truncated:
            break

    summary, steps = _summarize(tracker, scenario, spec)
    summary.update(seconds_policy=t_policy, seconds_env=t_env,
                   seconds_total=time.perf_counter() - t_start)
    if out_dir is not None:
        path = result_path(out_dir, spec, scenario)
        path.parent.mkdir(parents=True, exist_ok=True)
        if save_steps:
            import pandas as pd
            pd.DataFrame(steps).to_csv(path.with_suffix(".steps.csv.gz"), index=False, float_format="%.6g")
        path.write_text(json.dumps(summary, indent=2))
    return summary


def _summarize(tracker, scenario, spec):
    rewards = np.asarray(tracker.rewards, dtype=float)
    rc = tracker.reward_components
    regret = np.array([r["normalized_productivity_regret"] for r in rc], dtype=float)
    energy = np.array([r["normalized_energy_cost"] for r in rc], dtype=float)
    carbon = np.array([r["normalized_carbon_emission"] for r in rc], dtype=float)
    occ = np.asarray(tracker.occupancy, dtype=float)
    wp, we, wc = tracker.productivity_weight, tracker.energy_weight, tracker.carbon_weight
    comfort_s, energy_s, carbon_s = wp * np.abs(regret), we * energy, wc * carbon
    run = np.array([d["ahu_run_cmds"] for d in tracker.ahu_log], dtype=int)
    outside = np.array([d["outside_air"] for d in tracker.ahu_log], dtype=float)
    temps = np.asarray(tracker.zone_temps, dtype=float)
    T = len(rewards)
    occupied = occ > 0
    p = Path(scenario)
    summary = {
        "scenario": f"{p.parent.name}/{p.name}",
        "building": p.parent.name,
        "weather": p.name,
        "station": p.name.rsplit("_", 1)[0],
        "spec": spec,
        "steps": T,
        "total_reward": float(rewards.sum()),
        "score": float(abs(rewards.mean())) if T else float("nan"),
        "comfort_loss": float(comfort_s.mean()) if T else 0.0,
        "energy_loss": float(energy_s.mean()) if T else 0.0,
        "carbon_loss": float(carbon_s.mean()) if T else 0.0,
        "comfort_violation_frac": float((regret[occupied] != 0).mean()) if occupied.any() else 0.0,
        "zone_temp_min_occ_C": float(temps[occupied].min() - 273.15) if occupied.any() else None,
        "zone_temp_max_occ_C": float(temps[occupied].max() - 273.15) if occupied.any() else None,
        "n_zones": int(temps.shape[1]) if temps.ndim == 2 else None,
        "n_ahus": int(run.shape[1]) if run.ndim == 2 else None,
        "ahu_switches_per_ahu_day": (float(np.abs(np.diff(run, axis=0)).sum(0).mean() / (T / STEPS_PER_DAY))
                                     if run.ndim == 2 and T > 1 else None),
        "ahu_on_frac": float(run.mean()) if run.size else None,
    }
    steps = {
        "reward": rewards, "comfort": comfort_s, "energy": energy_s, "carbon": carbon_s,
        "occupancy": occ, "outside_C": outside - 273.15,
        "ahus_on": run.sum(1) if run.ndim == 2 else np.zeros(T),
        "zone_temp_min_C": temps.min(1) - 273.15 if temps.ndim == 2 else np.zeros(T),
        "zone_temp_max_C": temps.max(1) - 273.15 if temps.ndim == 2 else np.zeros(T),
    }
    return summary, steps


def _cached(out_dir, spec, scenario, steps) -> bool:
    p = result_path(out_dir, spec, scenario)
    if not p.exists():
        return False
    try:
        return json.loads(p.read_text()).get("steps") == steps
    except json.JSONDecodeError:
        return False


def _size(scenario) -> int:
    try:
        return int(np.load(paths.scenario_dir(scenario) / "floor_plan.npy", mmap_mode="r").size)
    except OSError:
        return 0


def run_batch(jobs, out_dir, workers=None, save_steps=False, verbose=True):
    """jobs: iterable of (scenario, spec, steps). Cached jobs are skipped; bigger buildings start first."""
    paths.setup()
    jobs = list(jobs)
    todo = [j for j in jobs if not _cached(out_dir, j[1], j[0], j[2])]
    size = {s: _size(s) for s in {j[0] for j in todo}}
    todo.sort(key=lambda j: -size[j[0]])
    workers = workers or int(os.getenv("SB_WORKERS", 0)) or os.cpu_count() or 1
    if verbose:
        print(f"{len(jobs) - len(todo)} cached, {len(todo)} to run on {workers} workers", flush=True)
    if not todo:
        return
    kw = {"max_workers": min(workers, len(todo)), "mp_context": mp.get_context("spawn")}
    if sys.version_info >= (3, 11):
        kw["max_tasks_per_child"] = 1
    t0 = time.time()
    with ProcessPoolExecutor(**kw) as ex:
        futs = {ex.submit(run_scenario, sc, spec, steps, str(out_dir), save_steps): (sc, spec)
                for sc, spec, steps in todo}
        for i, f in enumerate(as_completed(futs), 1):
            sc, spec = futs[f]
            try:
                s = f.result()
                if verbose:
                    print(f"[{i:3d}/{len(todo)}] {time.time() - t0:6.0f}s  {spec[:28]:28s} {sc:32s} "
                          f"score {s['score']:.4f}  ({s['seconds_total'] / 60:.1f} min)", flush=True)
            except Exception as e:  # keep the batch going; the job is retried next run
                print(f"[{i:3d}/{len(todo)}] FAILED {spec} {sc}: {e!r}", flush=True)


def load_results(out_dir, specs=None, steps=None):
    """All cached summaries under out_dir (optionally filtered) as a DataFrame."""
    import pandas as pd
    rows = []
    for p in Path(out_dir).glob("*/*.json"):
        try:
            r = json.loads(p.read_text())
        except json.JSONDecodeError:
            continue
        if "spec" not in r:
            continue
        if specs is not None and r["spec"] not in specs:
            continue
        if steps is not None and r["steps"] != steps:
            continue
        rows.append(r)
    return pd.DataFrame(rows)


def compare(df, ref_spec="ref", noise=None):
    """Per spec: mean score and change vs ref on the scenarios both ran.

    noise: DataFrame indexed by scenario with 'std' (from a null group). When
    given, adds per-scenario z = (score - ref) / std and counts |z| >= 2.
    """
    import pandas as pd
    piv = df.pivot_table(index="scenario", columns="spec", values="score")
    if ref_spec not in piv:
        raise KeyError(f"{ref_spec!r} has no results")
    rows = []
    for spec in piv.columns:
        both = pd.DataFrame({"ref": piv[ref_spec], "cand": piv[spec]}).dropna()
        if both.empty:
            continue
        d = both["cand"] - both["ref"]
        row = {"spec": spec, "scenarios": len(both), "score": both["cand"].mean(),
               "ref_score": both["ref"].mean(),
               "change_pct": 100 * d.sum() / both["ref"].sum(),
               "better": int((d < 0).sum()), "worse": int((d > 0).sum()),
               "worst_change_pct": float((100 * d / both["ref"]).max())}
        if noise is not None:
            sd = noise.reindex(both.index)["std"].replace(0, np.nan)
            z = (d / sd).dropna()
            row.update(z_better=int((z <= -2).sum()), z_worse=int((z >= 2).sum()))
        rows.append(row)
    return pd.DataFrame(rows).set_index("spec").sort_values("score")


In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sb_paths, sb_policy, sb_eval as ev
for m in (sb_paths, sb_policy, ev):
    importlib.reload(m)
DATA_ROOT = sb_paths.setup()
SCENARIOS = [s for s in sb_paths.training_scenarios() if not ONLY or s in ONLY]
print(f"data: {DATA_ROOT}\nscenarios: {len(SCENARIOS)}\nworkers: {WORKERS}")

## Chạy 30 kịch bản

In [ ]:
if RUN_EVAL:
    specs = ["cand"] + (["ref"] if RUN_REF else [])
    t0 = time.time()
    ev.run_batch([(s, sp, N_STEPS) for sp in specs for s in SCENARIOS], OUT, workers=WORKERS, save_steps=True)
    print(f"done in {(time.time() - t0) / 60:.1f} min")

## Điểm

In [ ]:
if RUN_EVAL:
    df = ev.load_results(OUT, steps=N_STEPS)
    df = df[df.scenario.isin(SCENARIOS)]
    cand = df[df.spec == "cand"].set_index("scenario").sort_values("score", ascending=False)
    table = cand[["score", "comfort_loss", "energy_loss", "carbon_loss"]].copy()

    base_file = DATA_ROOT / "baseline_returns.json"
    if base_file.exists() and N_STEPS == 8640:
        base = json.loads(base_file.read_text())
        table["pid_water"] = [abs(base[s]["pid_water"]) / 8640 if s in base else np.nan for s in table.index]
    ref = None
    if RUN_REF:
        ref = df[df.spec == "ref"].set_index("scenario").score
    elif REF_SCORES_CSV and Path(REF_SCORES_CSV).exists():
        ref = pd.read_csv(REF_SCORES_CSV).set_index("scenario").score
    if ref is not None:
        table["ref"] = ref.reindex(table.index)
        table["vs_ref_pct"] = 100 * (table.score - table.ref) / table.ref

    print(f"POLICY {POLICY}: score {table.score.mean():.5f} on {len(table)} scenarios")
    print(f"  comfort {table.comfort_loss.mean():.5f}  energy {table.energy_loss.mean():.5f}  carbon {table.carbon_loss.mean():.5f}")
    if "pid_water" in table:
        print(f"  pid_water on the same scenarios: {table.pid_water.mean():.5f}")
    if "ref" in table and table.ref.notna().all():
        print(f"  reference on the same scenarios: {table.ref.mean():.5f}  ({100 * (table.score.sum() / table.ref.sum() - 1):+.1f}%)")
    display(table.round(4))
    display(cand.groupby("station")[["score", "comfort_loss", "energy_loss", "carbon_loss"]].mean().round(4))

In [ ]:
if RUN_EVAL:
    fig, ax = plt.subplots(figsize=(10, 0.28 * len(table) + 1.2))
    y = np.arange(len(table))[::-1]
    left = np.zeros(len(table))
    for col, c in (("comfort_loss", "#2f6fb3"), ("energy_loss", "#d9551c"), ("carbon_loss", "#677689")):
        ax.barh(y, table[col], left=left, color=c, label=col.replace("_loss", ""))
        left += table[col].to_numpy()
    if "ref" in table:
        ax.scatter(table.ref, y, color="black", marker="|", s=120, label="reference", zorder=3)
    ax.set_yticks(y, table.index, fontsize=8)
    ax.set_xlabel("điểm (thấp hơn = tốt hơn)")
    ax.set_title(f"{POLICY}: {table.score.mean():.5f}")
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout(); fig.savefig(OUT / "scores.png", dpi=120); plt.show()

    table.to_csv(OUT / f"scores_{POLICY}.csv")
    summary = {"policy": POLICY, "steps": N_STEPS, "scenarios": len(table), "score": float(table.score.mean()),
               "comfort_loss": float(table.comfort_loss.mean()), "energy_loss": float(table.energy_loss.mean()),
               "carbon_loss": float(table.carbon_loss.mean())}
    (OUT / "summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))

## Nộp bài (Kaggle)

Chạy policy này qua inference server chính thức. Trên Kaggle, sau khi *Save Version* xong, bấm **Submit** để lấy điểm LB.

In [ ]:
if RUN_SERVER:
    for p in (str(DATA_ROOT), str(DATA_ROOT / "kaggle_evaluation")):
        if p not in sys.path:
            sys.path.insert(0, p)
    from kaggle_evaluation import smart_buildings_inference_server

    class SubmissionPolicy(sb_policy.CandidatePolicy, smart_buildings_inference_server.Policy):
        pass

    smart_buildings_inference_server.SmartBuildingsInferenceServer(SubmissionPolicy).run(
        competition_data_folder=str(DATA_ROOT))